# LUMOS evaluation

Load a trained checkpoint, run the model on the validation split, and compare the
recovered Raman spectrum to ground truth. The model is unsupervised; ground truth
is used only for scoring.

In [41]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from types import SimpleNamespace

from lumos.train import load_checkpoint
from lumos.datamodule import ZarrDataModule
from lumos.predict import sample_posterior
from lumos.metrics import spectral_angle_mapper, scale_invariant_psnr, nrmse

In [42]:
# Point these at your data and training run.
ZARR_PATH = ""
CHECKPOINT_DIR = "../checkpoints"   # relative to this notebook
RUN_NAME = None                     # None picks the most recent run

In [43]:
model, run_name = load_checkpoint(run_name=RUN_NAME, checkpoint_dir=CHECKPOINT_DIR)
model.eval()
frame_duration = model.model.frame_duration
print("loaded:", run_name)
print("physics:", model.model.physics_model,
      "| n_times_train:", model.hparams.n_times_train,
      "| frame_duration:", frame_duration)

FileNotFoundError: No checkpoints directory at ../checkpoints

In [ ]:
dm = ZarrDataModule(
    zarr_path=ZARR_PATH,
    config=SimpleNamespace(physics_model=model.model.physics_model),
    batch_size=8, num_workers=0,
    n_times_train=model.hparams.n_times_train, normalize=True,
)
dm.setup()
print("val samples:", len(dm.val_ds), "| ground truth:", "present" if dm.gt is not None else "none")

In [ ]:
# The Raman head outputs a rate (cts/s); multiply by the frame duration to get
# cts/frame, matching gt_raman.
n_eval = min(16, len(dm.val_ds))
wn = dm.gt['wavenumber'] if dm.gt is not None else dm.val_ds.wavenumbers
pred_ramans, gt_ramans = [], []
for i in range(n_eval):
    x = dm.val_ds[i][0].unsqueeze(0)  # [1, W, T]
    ens = sample_posterior(model, x, n_predictions=1, physics_model=model.model.physics_model)
    pred_ramans.append(ens['raman'].mean(axis=0) * frame_duration)
    if dm.gt is not None:
        gt_ramans.append(dm.gt['gt_raman'][i])
pred_ramans = np.stack(pred_ramans)
gt_ramans = np.stack(gt_ramans) if gt_ramans else None

In [ ]:
if gt_ramans is not None:
    sam = [spectral_angle_mapper(g, p) for g, p in zip(gt_ramans, pred_ramans)]
    psnr = [scale_invariant_psnr(g, p) for g, p in zip(gt_ramans, pred_ramans)]
    nr = [nrmse(g, p) for g, p in zip(gt_ramans, pred_ramans)]
    print(f'SAM      {np.mean(sam):.4f}')
    print(f'SI-PSNR  {np.mean(psnr):.2f} dB')
    print(f'NRMSE    {np.mean(nr):.4f}')
else:
    print('no ground truth in this store; skipping metrics')

In [ ]:
k = min(10, n_eval)
fig, axes = plt.subplots(k, 1, figsize=(8, 2.2 * k), sharex=True)
axs = np.atleast_1d(axes)
for i in range(k):
    if gt_ramans is not None:
        axs[i].plot(wn, gt_ramans[i], lw=1.0, label='ground truth')
    axs[i].plot(wn, pred_ramans[i], lw=1.0, label='recovered')
    axs[i].set_ylabel('cts/frame')
    axs[i].legend(loc='upper right', fontsize=8)
axs[-1].set_xlabel('wavenumber (cm^-1)')
plt.tight_layout()
plt.show()